# Treinamento do Modelo

- Treinar modelo
- Exportar NIR

# Exportar Dataset de Teste

- Salvar dataset como .npz
  - chave 'data' para os dados
  - chave 'labels' para os rótulos

## Exemplo com Torch

In [1]:
# import torch
# from torchvision import datasets, transforms
# from torch.utils.data import DataLoader
# import numpy as np

# # Transformação: converte PIL Images para tensores e normaliza
# transform = transforms.Compose([
#     transforms.ToTensor(),             # converte para torch.Tensor
#     transforms.Normalize((0.1307,), (0.3081,))  # normalização MNIST
# ])

# # Baixa MNIST de treino
# mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# # Cria DataLoader
# dataloader_torch = DataLoader(mnist_train, batch_size=1, shuffle=True)

In [2]:
# def salvar_dataset_npz(loader, arquivo):
#     dados = []
#     labels = []

#     for batch in loader:
#         # formato (x, y)
#         x, y = batch

#         # PyTorch
#         if torch.is_tensor(x):
#             x = x.cpu().numpy()
#             y = y.cpu().numpy()

#         dados.append(x)
#         labels.append(y)

#     dados = np.concatenate(dados, axis=0)
#     labels = np.concatenate(labels, axis=0)

#     np.savez_compressed(
#         arquivo,
#         data=dados,
#         labels=labels
#     )

#     print(f"Arquivo '{arquivo}.npz' salvo.")

In [3]:
# salvar_dataset_npz(dataloader_torch, "data_test.npz")

## Exemplo com Tensorflow

- TODO: FINALIZAR

In [4]:
# import tensorflow as tf

# # Carrega MNIST direto do Keras datasets
# (mnist_x_train, mnist_y_train), _ = tf.keras.datasets.mnist.load_data()

# # Normaliza os pixels
# mnist_x_train = mnist_x_train.astype('float32') / 255.0
# mnist_x_train = mnist_x_train[..., tf.newaxis]  # adiciona canal (28,28,1)

# # Converte labels para tf.int32
# mnist_y_train = mnist_y_train.astype('int32')

# # Cria tf.data.Dataset
# dataset_tf = tf.data.Dataset.from_tensor_slices((mnist_x_train, mnist_y_train))
# dataset_tf = dataset_tf.shuffle(buffer_size=10000).batch(32)

In [5]:
# import numpy as np

# def processar_batch(batch_x, batch_y):
#     """
#     Função genérica para processar um batch de dados.
#     Aqui você pode colocar treino, validação, ou qualquer operação.
#     """
#     print("Batch X shape:", batch_x.shape)
#     print("Batch Y shape:", batch_y.shape)
#     # Exemplo de operação simples
#     return batch_x.mean(), batch_y.mean()

# def iterar_dataloader(dataloader):
#     """
#     Itera sobre qualquer dataloader PyTorch ou TensorFlow,
#     convertendo para numpy arrays.
#     """
#     for batch_x, batch_y in dataloader:
#         # PyTorch e TensorFlow 2.x retornam tensors com .numpy()
#         if hasattr(batch_x, "numpy"):
#             batch_x = batch_x.numpy()
#             batch_y = batch_y.numpy()
#         # Agora batch_x e batch_y são np.array
#         yield batch_x, batch_y
        
# def main(dataloader_torch, dataloader_tf):
#     print("Iterando DataLoader do PyTorch")
#     for batch_x, batch_y in iterar_dataloader(dataloader_torch):
#         processar_batch(batch_x, batch_y)
    
#     print("\nIterando DataLoader do TensorFlow")
#     for batch_x, batch_y in iterar_dataloader(dataloader_tf):
#         processar_batch(batch_x, batch_y)

In [6]:
# main(dataloader_torch, dataset_tf)

# NeuroHls

In [1]:
from neuro_hls import *

In [2]:
neuro_hls = NeuroHls("z_test")

## Definindo a Implementação do Modelo

In [3]:
# nir_file = "nir_examples/lif_norse.nir"
nir_file = "nir_examples/cnn_sinabs.nir"
# nir_file = "nir_examples/braille_noDelay_bias_zero.nir"

In [4]:
model = neuro_hls.read_nir_file(nir_file)

In [5]:
print(model)

-------------------------------------------------------
Input ([ 2 34 34]) - layer name: 'input'
-------------------------------------------------------
	Is recurrent: NO
	Dependencies:

-------------------------------------------------------
Conv2d (input: [ 2 34 34], output: [16 16 16]) - layer name: '0'
-------------------------------------------------------
	Weight shape: (16, 2, 5, 5)
	Stride: [2 2], Padding: [1 1], Dilation: [1 1]
	Groups: 1, Bias shape: (16,)
	Is recurrent: NO
	Dependencies:
	   - input (ready)

-------------------------------------------------------
IF (input: [16 16 16], output: [16 16 16]) - layer name: '1'
-------------------------------------------------------
	Parameter shape: (16, 16, 16)
	r range: [1.0000, 1.0000]
	v_threshold range: [1.0000, 1.0000]
	v_reset range: [0.0000, 0.0000]
	Is recurrent: NO
	Dependencies:
	   - 0 (ready)

-------------------------------------------------------
Conv2d (input: [16 16 16], output: [16 16 16]) - layer name: '2'
---

In [6]:
neuro_hls.implement_model(model)

## Criando o Testbench

In [13]:
neuro_hls.define_test_dataset("nir_examples/rnn_test.pt", data_is_binary=True, step_count=256, different_sample_per_step=True)

In [14]:
neuro_hls.create_testbench(total_samples=140, batch_size=5)

Total samples used: 140 of 140
Batch size: 5
Total batches: 28
Testbench was created.


In [15]:
# neuro_hls.run_csim()